# Bayesian Spatial Durbin Model (BSDM)
This notebook implements the Bayesian Spatial Durbin Model for the Silk Road tourism dataset,
including model estimation, diagnostic checks, and robustness tests (LOO and Moran’s I).


In [ ]:
!pip install libpysal esda

In [ ]:
!pip install geopy

In [3]:
!pip install pymc arviz

In [4]:
import pymc as pm
import arviz as az
import numpy as np
import pandas as pd

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


In [5]:
df_sorted = pd.read_excel('FINAL_WIDE_W.xlsx')
df_sorted

FileNotFoundError: [Errno 2] No such file or directory: 'FINAL_WIDE_W.xlsx'

In [ ]:
cities = {
    'Azerbaijan': (40.4093, 49.8671),     # Baku
    'China': (34.3416, 108.9398),         # Xi'an
    'Egypt': (31.2001, 29.9187),          # Alexandria
    'Georgia': (42.2679, 42.7180),        # Kutaisi
    'India': (27.0920, 77.6689),          # Fatehpur Sikri
    'Iran': (35.5760, 53.3950),           # Semnan
    'Italy': (45.4408, 12.3155),          # Venice
    'Kazakhstan': (43.2220, 76.8512),     # Almaty
    'Kyrgyzstan': (41.1000, 75.3500),     # Tash Rabat
    'Saudi Arabia': (21.4858, 39.1925),   # Jeddah
    'Türkiye': (40.1828, 29.0666),         # Bursa
}

In [ ]:
from geopy.distance import geodesic
import numpy as np

# --- Step 1: Define the list of countries in a consistent order ---
country_list = sorted(cities.keys())
N = len(country_list)

# --- Step 2: Extract geographic coordinates for each country ---
coords = [cities[c] for c in country_list]

# --- Step 3: Construct the distance matrix ---
# Compute pairwise great-circle distances (in kilometers) using geodesic distance
dist_matrix = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        if i != j:
            dist_matrix[i, j] = geodesic(coords[i], coords[j]).kilometers
        else:
            dist_matrix[i, j] = np.inf  # avoid division by zero on the diagonal

# --- Step 4: Construct the inverse-distance weight matrix ---
W_distance_inv = 1 / dist_matrix

# --- Step 5: Row-standardize the matrix (so that each row sums to 1) ---
W_distance_inv = W_distance_inv / W_distance_inv.sum(axis=1, keepdims=True)

# --- Step 6: Store the weight matrix for each year ---
# Since geographic distances are time-invariant, the same matrix is assigned to all years
W_dict = {year: W_distance_inv for year in df_sorted['year'].unique()}



✅ Final step: Run the Bayesian SDM with the inverse-distance weight matrix

In [ ]:

# Assume: df_sorted, X, y, W_dict, countries, and years have already been prepared.
# X represents the matrix of independent variables, for example:
X = df_sorted[['GDP', 'Political Stability', 'exchange rate', "Rule of Law: Estimate"]].values

K = X.shape[1]  # Number of independent variables
N = df_sorted.shape[0]  # Total number of data rows (observations)
y = df_sorted['inbound'].values  # Dependent variable

# Map country and year indices
country_to_idx = {c: i for i, c in enumerate(sorted(df_sorted['country'].unique()))}
year_to_idx = {y: i for i, y in enumerate(sorted(df_sorted['year'].unique()))}

N = len(country_to_idx)
T = len(year_to_idx)
K = X.shape[1]

# Construct Wy and WX
y_neighbors = []
WX = np.zeros((N * T, K))

for idx, row in df_sorted.iterrows():
    year = row['year']
    country = row['country']
    c_idx = country_to_idx[country]
    y_idx = year_to_idx[year]
    obs_idx = y_idx * N + c_idx

    # Weight matrix for the current year
    W = W_dict[year]

    # y values for the current year
    y_t = (
        df_sorted[df_sorted['year'] == year]
        .set_index('country')
        .loc[sorted(df_sorted['country'].unique())]['inbound']
        .values
    )

    # Compute Wy
    wy = np.dot(W[c_idx], y_t)
    y_neighbors.append(wy)

    # Compute WX (weighted independent variables)
    X_t = X[y_idx * N:(y_idx + 1) * N]
    WX[obs_idx] = np.dot(W[c_idx], X_t)

y_neighbors = np.array(y_neighbors)

# Country index for fixed effects
country_idx = pd.Categorical(df_sorted['country']).codes

# ✅ Build the Bayesian Spatial Durbin Model (SDM) using PyMC
with pm.Model() as sdm_model:
    # Priors
    beta = pm.Normal("beta", mu=0, sigma=10, shape=K)
    gamma = pm.Normal("gamma", mu=0, sigma=10, shape=K)
    alpha = pm.Normal("alpha", mu=0, sigma=1, shape=N)
    rho = pm.Normal("rho", mu=0, sigma=1)
    sigma = pm.HalfCauchy("sigma", beta=2)

    # Spatial regression equation
    mu = (
        pm.math.dot(X, beta)
        + pm.math.dot(WX, gamma)
        + alpha[country_idx]
        + rho * y_neighbors
    )

    # Likelihood for the dependent variable
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)

    # Sampling
    trace = pm.sample(100, tune=100, target_accept=0.9, return_inferencedata=True)


✅ Next Step: Summarizing and Visualizing the Results




In [ ]:
# --- Parameter summary ---
summary = az.summary(trace, var_names=["rho", "beta", "gamma", "sigma", "alpha"], round_to=4)
print(summary)

# --- Diagnostic plots ---
az.plot_trace(trace, var_names=["rho", "beta", "gamma"])
az.plot_posterior(trace, var_names=["rho", "gamma"])

summary

In [ ]:
import matplotlib.pyplot as plt
import arviz as az
import numpy as np

# Extract country-level effects (alpha) from the trace
alpha_samples = trace.posterior["alpha"].stack(draws=("chain", "draw")).values  # shape: (N countries, samples)

# Compute mean and HDI (Highest Density Interval)
alpha_means = alpha_samples.mean(axis=1)
alpha_hdi = az.hdi(alpha_samples.T, hdi_prob=0.94)  # shape: (N countries, 2)

# Prepare data for visualization
countries_ordered = sorted(country_to_idx.keys())
lower_bounds = alpha_hdi[:, 0]
upper_bounds = alpha_hdi[:, 1]

# Plot the bar chart with HDI error bars
fig, ax = plt.subplots(figsize=(10, 5))
errors = [alpha_means - lower_bounds, upper_bounds - alpha_means]
ax.bar(countries_ordered, alpha_means, yerr=errors, capsize=5, color='steelblue', alpha=0.7)

# Aesthetic adjustments
ax.set_ylabel("Alpha (Country Effects)")
ax.set_title("Country Effects (Alpha) with 94% HDI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(False)
plt.show()


# Model Robustness Test

This section evaluates the robustness and adequacy of the spatial specification in the Bayesian Spatial Durbin Model (BSDM).  
The robustness assessment is based on predictive performance and residual spatial diagnostics.

---

## 1. Predictive Comparison (LOO)
The Leave-One-Out cross-validation (LOO) statistics are used to evaluate the model’s out-of-sample predictive accuracy.  
Higher `elpd_loo` values (less negative) indicate better model performance and more stable spatial dependence structures, confirming the consistency of the estimated parameters across alternative spatial specifications.


---

## 2. Residual Spatial Autocorrelation (Moran’s I)
Moran’s I statistic is applied to assess the presence of spatial autocorrelation in the model residuals.  
A non-significant Moran’s I value indicates that the spatial dependence has been adequately captured by the BSDM, implying that no systematic spatial pattern remains unexplained in the residuals.


---

**Conclusion:**  
Across all robustness checks performed, the spatial dependence captured by the BSDM remains statistically consistent and theoretically sound, confirming that the model adequately represents spatial interactions along the Silk Road network.





In [ ]:
with sdm_model:
    trace = pm.sample(
        1000, tune=1000, target_accept=0.9, chains=2, cores=2,
        random_seed=42, return_inferencedata=True,
        idata_kwargs={"log_likelihood": True}  # <-- this is the key
    )

In [ ]:
loo_base = az.loo(trace, pointwise=False)
print(loo_base)

In [ ]:
import numpy as np
import pandas as pd
from libpysal.weights import W
from esda.moran import Moran


In [ ]:
# Mean of posterior predictions
y_pred = trace.posterior["beta"].mean(dim=("chain", "draw")).values @ X.T \
          + trace.posterior["gamma"].mean(dim=("chain", "draw")).values @ WX.T \
          + trace.posterior["rho"].mean().values * y_neighbors \
          + trace.posterior["alpha"].mean(dim=("chain", "draw")).values[country_idx]

# Compute residuals
residuals = y - y_pred


In [ ]:
from libpysal.weights import W
from libpysal.weights.util import full2W

# Assume W_dict[2019] is a NumPy matrix of size (N×N)
W_geo = full2W(W_dict[2019])  # Convert from NumPy array → PySAL W object

# Row-standardization
W_geo.transform = 'R'  # Row-standardize (so each row sums to 1)

In [ ]:
import numpy as np
import pandas as pd
from libpysal.weights.util import full2W
from esda.moran import Moran

# 1) Ensure consistent ordering
countries_sorted = sorted(df_sorted['country'].unique())
years_sorted     = sorted(df_sorted['year'].unique())

# 2) If residuals are already aligned with df_sorted:
#    It's better to store them as a column for easy filtering by year and country
df_sorted = df_sorted.sort_values(['year', 'country']).reset_index(drop=True)

# Residuals must have exactly the same length and order as df_sorted
df_sorted['residual'] = residuals  # vector length = N * T

# 3) Compute Moran’s I for a specific year (e.g., 2019)
year = 2019

# Extract the residuals vector in the order of countries_sorted
r_y = (
    df_sorted[df_sorted['year'] == year]
    .set_index('country')
    .loc[countries_sorted]['residual']
    .values
)

# Get the weight matrix for the same year, convert it to a PySAL W object, and row-standardize it
W_mat = W_dict[year]  # NumPy array (N×N), matching the order of countries_sorted
W_y   = full2W(W_mat)
W_y.transform = 'R'  # Row-standardize

# Moran’s I calculation
m_y = Moran(r_y, W_y)
print(f"Year {year} -> Moran’s I = {m_y.I:.3f}, p-value = {m_y.p_sim:.4f}")



---
**Repository:** [Silk-Road-Tourism-Spillovers](https://github.com/FatemehRafiei/Silk-Road-Tourism-Spillovers)  
**Author:** Fatemeh Rafiei (2025)
